In [2]:
import os
import re
import cv2
import dlib
import shutil
import random
import numpy as np
import pandas as pd

from PIL import Image
from datetime import datetime
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor, as_completed
from albumentations import (Compose, HorizontalFlip, RandomBrightnessContrast,
                            Rotate, Affine, GaussianBlur, GaussNoise, RGBShift,
                            OpticalDistortion, Perspective, Equalize)

In [3]:
VIDEO_ROOT = "/mnt/d/lip_codebase/Data for lip reading model/Videos of 10 selected words named"

# Location mapping, derived from the descriptive folder names in the archive.
# Keys are set numbers (1-60), values are (location, angle, zoom).
session_info = {
    1:  ("unlabelled",     "unknown", "no"),
    2:  ("owen",           "unknown", "no"),
    3:  ("lib_L2",         "normal",  "no"),
    4:  ("lib_L2",         "up",      "no"),
    5:  ("lib_L2",         "left",    "no"),
    6:  ("lib_L2",         "right",   "no"),
    7:  ("room_lit",       "normal",  "no"),
    8:  ("room_lit",       "up",      "no"),
    9:  ("room_lit",       "left",    "no"),
    10: ("room_lit",       "right",   "no"),
    11: ("unlabelled",     "unknown", "no"),
    12: ("lib_L4",         "normal",  "no"),
    13: ("lib_L4",         "up",      "no"),
    14: ("lib_L4",         "left",    "no"),
    15: ("lib_L4",         "right",   "no"),
    16: ("owen",           "unknown", "no"),
    17: ("room_natural",   "normal",  "no"),
    18: ("room_natural",   "up",      "no"),
    19: ("room_natural",   "left",    "no"),
    20: ("room_natural",   "right",   "no"),
    21: ("lib_L4",         "normal",  "zoom1"),
    22: ("lib_L4",         "up",      "zoom1"),
    23: ("lib_L4",         "left",    "zoom1"),
    24: ("lib_L4",         "right",   "zoom1"),
    25: ("lib_L4",         "down",    "zoom1"),
    26: ("lib_L2",         "normal",  "zoom1"),
    27: ("lib_L2",         "up",      "zoom1"),
    28: ("lib_L2",         "left",    "zoom1"),
    29: ("lib_L2",         "right",   "zoom1"),
    30: ("lib_L2",         "down",    "zoom1"),
    31: ("room_lit",       "normal",  "zoom1"),
    32: ("room_lit",       "up",      "zoom1"),
    33: ("room_lit",       "left",    "zoom1"),
    34: ("room_lit",       "right",   "zoom1"),
    35: ("room_lit",       "down",    "zoom1"),
    36: ("room_natural",   "normal",  "zoom1"),
    37: ("room_natural",   "up",      "zoom1"),
    38: ("room_natural",   "left",    "zoom1"),
    39: ("room_natural",   "right",   "zoom1"),
    40: ("room_natural",   "down",    "zoom1"),
    41: ("room_lit",       "normal",  "zoom1"),
    42: ("room_lit",       "up",      "zoom1"),
    43: ("room_lit",       "left",    "zoom1"),
    44: ("room_lit",       "right",   "zoom1"),
    45: ("room_lit",       "down",    "zoom1"),
    46: ("room_lit",       "normal",  "zoom1"),
    47: ("room_lit",       "up",      "zoom1"),
    48: ("room_lit",       "left",    "zoom1"),
    49: ("room_lit",       "right",   "zoom1"),
    50: ("room_lit",       "down",    "zoom1"),
    51: ("lib_L2_natural", "normal",  "zoom1"),
    52: ("lib_L2_natural", "up",      "zoom1"),
    53: ("lib_L2_natural", "left",    "zoom1"),
    54: ("lib_L2_natural", "right",   "zoom1"),
    55: ("lib_L2_natural", "down",    "zoom1"),
    56: ("harmer",         "normal",  "zoom1"),
    57: ("harmer",         "up",      "zoom1"),
    58: ("harmer",         "left",    "zoom1"),
    59: ("harmer",         "right",   "zoom1"),
    60: ("harmer",         "down",    "zoom1"),
}

df = pd.DataFrame(
    [(n, f"set_{n:02d}", loc, ang, z) for n, (loc, ang, z) in session_info.items()],
    columns=["set_num", "folder", "location", "angle", "zoom"]
).sort_values("set_num").reset_index(drop=True)

print("mapped sessions:", len(df))
print()
print(df.groupby("location").size().sort_values(ascending=False))
print()

# Read-only verification against the actual folders
actual = sorted(os.listdir(VIDEO_ROOT))
expected = [f"set_{n:02d}" for n in range(1, 61)]
print("folders found:", len(actual))
print("missing from disk:", [f for f in expected if f not in actual])
print("unexpected on disk:", [f for f in actual if f not in expected])

# Confirm every folder has 10 videos
bad = []
for f in expected:
    vids = [v for v in os.listdir(os.path.join(VIDEO_ROOT, f)) if v.endswith(".mp4")]
    if len(vids) != 10:
        bad.append((f, len(vids)))
print("folders without exactly 10 videos:", bad if bad else "none")

mapped sessions: 60

location
room_lit          19
lib_L2             9
room_natural       9
lib_L4             9
lib_L2_natural     5
harmer             5
owen               2
unlabelled         2
dtype: int64

folders found: 60
missing from disk: []
unexpected on disk: []
folders without exactly 10 videos: none


In [4]:
SEED = 4

quotas = {
    "room_lit":       (13, 3, 3),
    "room_natural":   (7,  1, 1),
    "lib_L2":         (6,  1, 2),
    "lib_L4":         (7,  1, 1),
    "lib_L2_natural": (3,  1, 1),
    "harmer":         (3,  1, 1),
    "owen":           (2,  0, 0),
    "unlabelled":     (1,  1, 0),
}

rng = random.Random(SEED)
assignment = {}

for loc, (n_tr, n_va, n_te) in quotas.items():
    sets = sorted(df.loc[df.location == loc, "set_num"].tolist())
    rng.shuffle(sets)
    for s in sets[:n_tr]:                       assignment[s] = "train"
    for s in sets[n_tr:n_tr+n_va]:              assignment[s] = "validation"
    for s in sets[n_tr+n_va:n_tr+n_va+n_te]:    assignment[s] = "test"

df["split"] = df.set_num.map(assignment)

print(df.split.value_counts())
print()
print(pd.crosstab(df.location, df.split))
print()
print(pd.crosstab(df.angle, df.split))
print()
for sp in ["train", "validation", "test"]:
    print(sp, sorted(df.loc[df.split == sp, "set_num"].tolist()))

split
train         42
validation     9
test           9
Name: count, dtype: int64

split           test  train  validation
location                               
harmer             1      3           1
lib_L2             2      6           1
lib_L2_natural     1      3           1
lib_L4             1      7           1
owen               0      2           0
room_lit           3     13           3
room_natural       1      7           1
unlabelled         0      1           1

split    test  train  validation
angle                           
down        2      5           1
left        1      9           2
normal      3      8           1
right       2      8           2
unknown     0      3           1
up          1      9           2

train [1, 2, 3, 4, 5, 6, 7, 8, 12, 13, 14, 16, 17, 19, 20, 21, 22, 24, 25, 28, 29, 31, 32, 33, 35, 37, 38, 39, 40, 42, 43, 45, 46, 47, 48, 49, 52, 54, 55, 56, 58, 59]
validation [9, 11, 15, 27, 36, 44, 50, 53, 57]
test [10, 18, 23, 26, 30, 34, 41, 51

In [5]:
OUT_DIR = "/home/admins/lip_codebase_clean/docs/results_rebuilt_data"
os.makedirs(OUT_DIR, exist_ok=True)

df.to_csv(os.path.join(OUT_DIR, "session_split_assignment.csv"), index=False)

with open(os.path.join(OUT_DIR, "split_method.txt"), "w") as f:
    f.write("Session-based split for rebuilt dataset\n")
    f.write("=======================================\n\n")
    f.write("Source: 60 recording sessions, 10 words each (600 videos)\n")
    f.write("Split: 42 train / 9 validation / 9 test sessions\n\n")
    f.write("Method: stratified random assignment by recording location,\n")
    f.write("using random.Random(seed).shuffle within each location group.\n\n")
    f.write("Seed: 4\n\n")
    f.write("Seed selection: seeds were checked against a fixed criterion set\n")
    f.write("before searching - that all five head angles (normal, up, left,\n")
    f.write("right, down) appear in both the test and validation splits.\n")
    f.write("Angle was not stratified directly because 9 test sessions across\n")
    f.write("5 angles and 8 locations gives strata too small to fill.\n")
    f.write("Seeds 42, 7, 3, 11, 5, 13, 17 and 4 were examined; seed 4 gave\n")
    f.write("the most even angle distribution meeting the criterion.\n")
    f.write("The criterion concerns condition coverage only, not model accuracy.\n\n")
    f.write("Rule: augmented images are generated only from sessions within\n")
    f.write("their own split. No augmented image derives from a session\n")
    f.write("assigned to a different split.\n")

print("saved to", OUT_DIR)
print(df.split.value_counts())

saved to /home/admins/lip_codebase_clean/docs/results_rebuilt_data
split
train         42
validation     9
test           9
Name: count, dtype: int64


In [6]:
VIDEO_ROOT = "/mnt/d/lip_codebase/Data for lip reading model/Videos of 10 selected words named"
WORKSPACE  = "/home/admins/rebuild_workspace"
FRAMES_DIR = os.path.join(WORKSPACE, "01_frames_original")
LOGS_DIR   = os.path.join(WORKSPACE, "logs")

TARGET_FRAMES = 60
WORDS = ['bat', 'cup', 'drop', 'eat', 'fish', 'hot', 'jump', 'milk', 'pen', 'red']

os.makedirs(LOGS_DIR, exist_ok=True)


def extract_frames(video_path, output_dir, target_frames=TARGET_FRAMES):
    cap = cv2.VideoCapture(video_path)
    frames = []
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frames.append(frame)
    cap.release()

    if not frames:
        return None

    source_count = len(frames)
    padded = trimmed_start = trimmed_end = 0

    if source_count < target_frames:
        padded = target_frames - source_count
        frames = frames + [frames[-1]] * padded
    elif source_count > target_frames:
        excess = source_count - target_frames
        trimmed_end = int(excess * 0.8)
        trimmed_start = excess - trimmed_end
        frames = frames[trimmed_start:source_count - trimmed_end]

    os.makedirs(output_dir, exist_ok=True)
    for i, f in enumerate(frames, start=1):
        cv2.imwrite(os.path.join(output_dir, f"{i:02d}.png"), f)

    h, w = frames[0].shape[:2]
    return dict(source_frames=source_count, written=len(frames),
                padded=padded, trimmed_start=trimmed_start,
                trimmed_end=trimmed_end, height=h, width=w)


records = []
start = datetime.now()

for n in range(1, 61):
    set_name = f"set_{n:02d}"
    src_dir = os.path.join(VIDEO_ROOT, set_name)

    for word in WORDS:
        video_path = os.path.join(src_dir, f"{word}.mp4")
        if not os.path.exists(video_path):
            print(f"MISSING: {set_name}/{word}.mp4")
            continue

        out_dir = os.path.join(FRAMES_DIR, set_name, word)
        result = extract_frames(video_path, out_dir)

        if result is None:
            print(f"FAILED: {set_name}/{word}.mp4")
            continue

        records.append(dict(set_num=n, set_name=set_name, word=word, **result))

    print(f"{set_name} done  ({(datetime.now()-start).seconds}s elapsed)")

log = pd.DataFrame(records)
log.to_csv(os.path.join(LOGS_DIR, "frame_extraction_log.csv"), index=False)

print("\n--- summary ---")
print("videos processed:", len(log))
print("all wrote 60 frames:", (log.written == 60).all())
print("\nsource frame counts:")
print(log.source_frames.describe())
print("\npadded videos:", (log.padded > 0).sum())
print("trimmed videos:", (log.trimmed_end > 0).sum())
print("exact videos:", ((log.padded == 0) & (log.trimmed_end == 0)).sum())
print("\nresolutions:", log.groupby(['height','width']).size().to_dict())

set_01 done  (21s elapsed)
set_02 done  (46s elapsed)
set_03 done  (70s elapsed)
set_04 done  (94s elapsed)
set_05 done  (119s elapsed)
set_06 done  (142s elapsed)
set_07 done  (162s elapsed)
set_08 done  (183s elapsed)
set_09 done  (202s elapsed)
set_10 done  (221s elapsed)
set_11 done  (242s elapsed)
set_12 done  (265s elapsed)
set_13 done  (287s elapsed)
set_14 done  (310s elapsed)
set_15 done  (333s elapsed)
set_16 done  (358s elapsed)
set_17 done  (378s elapsed)
set_18 done  (398s elapsed)
set_19 done  (417s elapsed)
set_20 done  (436s elapsed)
set_21 done  (460s elapsed)
set_22 done  (484s elapsed)
set_23 done  (509s elapsed)
set_24 done  (533s elapsed)
set_25 done  (558s elapsed)
set_26 done  (580s elapsed)
set_27 done  (603s elapsed)
set_28 done  (625s elapsed)
set_29 done  (647s elapsed)
set_30 done  (670s elapsed)
set_31 done  (690s elapsed)
set_32 done  (710s elapsed)
set_33 done  (730s elapsed)
set_34 done  (751s elapsed)
set_35 done  (771s elapsed)
set_36 done  (796s elaps

In [7]:
WORKSPACE   = "/home/admins/rebuild_workspace"
FRAMES_DIR  = os.path.join(WORKSPACE, "01_frames_original")
CROPPED_DIR = os.path.join(WORKSPACE, "02_frames_cropped")
LOGS_DIR    = os.path.join(WORKSPACE, "logs")
PREDICTOR   = "/home/admins/lip_codebase_clean/models/shape_predictor_68_face_landmarks.dat"

LIP_HEIGHT, LIP_WIDTH = 80, 112
WORDS = ['bat', 'cup', 'drop', 'eat', 'fish', 'hot', 'jump', 'milk', 'pen', 'red']

_detector = None
_predictor = None

def _init_worker():
    global _detector, _predictor
    _detector = dlib.get_frontal_face_detector()
    _predictor = dlib.shape_predictor(PREDICTOR)


def crop_lip(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = _detector(gray)
    if not faces:
        return None
    landmarks = _predictor(gray, faces[0])
    mouth = np.array([(landmarks.part(n).x, landmarks.part(n).y) for n in range(48, 68)])
    x, y, w, h = cv2.boundingRect(mouth)

    pad_left   = max((LIP_WIDTH - w) // 2, 0)
    pad_right  = max((LIP_WIDTH - w) - pad_left, 0)
    pad_top    = max((LIP_HEIGHT - h) // 2, 0)
    pad_bottom = max((LIP_HEIGHT - h) - pad_top, 0)

    pad_left   = min(pad_left, x)
    pad_right  = min(pad_right, frame.shape[1] - (x + w))
    pad_top    = min(pad_top, y)
    pad_bottom = min(pad_bottom, frame.shape[0] - (y + h))

    lip = frame[y - pad_top:y + h + pad_bottom, x - pad_left:x + w + pad_right]
    if lip.size == 0:
        return None
    return cv2.resize(lip, (LIP_WIDTH, LIP_HEIGHT))


def fill_gaps(crops):
    n = len(crops)
    if not any(c is not None for c in crops):
        return None, None
    filled, records, i = list(crops), [], 0
    while i < n:
        if filled[i] is not None:
            i += 1
            continue
        start = i
        while i < n and crops[i] is None:
            i += 1
        end, length = i - 1, i - start
        before = start - 1 if start > 0 else None
        after  = i if i < n else None
        if before is None:
            for k in range(start, end + 1):
                filled[k] = crops[after]; records.append((k + 1, after + 1))
        elif after is None:
            for k in range(start, end + 1):
                filled[k] = crops[before]; records.append((k + 1, before + 1))
        else:
            first_half = (length + 1) // 2
            for offset, k in enumerate(range(start, end + 1)):
                src = before if offset < first_half else after
                filled[k] = crops[src]; records.append((k + 1, src + 1))
    return filled, records


def process_word(args):
    set_name, word = args
    src = os.path.join(FRAMES_DIR, set_name, word)
    dst = os.path.join(CROPPED_DIR, set_name, word)
    frame_files = sorted(f for f in os.listdir(src) if f.endswith('.png'))

    crops = []
    for f in frame_files:
        frame = cv2.imread(os.path.join(src, f))
        crops.append(None if frame is None else crop_lip(frame))

    failed = [i + 1 for i, c in enumerate(crops) if c is None]
    filled, records = fill_gaps(crops)

    if filled is None:
        return dict(set_name=set_name, word=word, total=len(crops), detected=0,
                    failed_frames=str(failed), fills="ALL FAILED", status="TOTAL_FAILURE")

    os.makedirs(dst, exist_ok=True)
    for i, c in enumerate(filled, start=1):
        cv2.imwrite(os.path.join(dst, f"{i:02d}.png"), c)

    return dict(set_name=set_name, word=word, total=len(crops),
                detected=len(crops) - len(failed),
                failed_frames=str(failed) if failed else "",
                fills=str(records) if records else "",
                status="OK" if not failed else "FILLED")


if True:
    tasks = [(f"set_{n:02d}", w) for n in range(1, 61) for w in WORDS]
    records, start, done = [], datetime.now(), 0

    with ProcessPoolExecutor(max_workers=10, initializer=_init_worker) as ex:
        futures = {ex.submit(process_word, t): t for t in tasks}
        for fut in as_completed(futures):
            r = fut.result()
            records.append(r)
            done += 1
            if r['status'] != "OK":
                print(f"  {r['set_name']}/{r['word']}: {r['status']} "
                      f"({r['detected']}/60) frames {r['failed_frames']}")
            if done % 100 == 0:
                print(f"{done}/600  ({(datetime.now()-start).seconds}s)")

    log = pd.DataFrame(records).sort_values(['set_name', 'word']).reset_index(drop=True)
    log.to_csv(os.path.join(LOGS_DIR, "crop_detection_log.csv"), index=False)

    print("\n--- summary ---")
    print("videos processed:", len(log))
    print("fully detected  :", (log.status == "OK").sum())
    print("needed filling  :", (log.status == "FILLED").sum())
    print("total failures  :", (log.status == "TOTAL_FAILURE").sum())
    print("elapsed:", (datetime.now()-start).seconds, "s")

100/600  (61s)
200/600  (121s)
300/600  (180s)
400/600  (240s)
500/600  (300s)
600/600  (360s)

--- summary ---
videos processed: 600
fully detected  : 600
needed filling  : 0
total failures  : 0
elapsed: 360 s


In [8]:
df = pd.read_csv("/home/admins/lip_codebase_clean/docs/results_rebuilt_data/session_split_assignment.csv")
print(df.split.value_counts())

split
train         42
validation     9
test           9
Name: count, dtype: int64


In [9]:
AUG_SEED = 101

STOCHASTIC = ["Rotate", "Translate", "Scale", "GaussianBlur",
              "GaussNoise", "RGBShift", "OpticalDistortion", "Perspective"]

FIXED = ["HorizontalFlip", "Brightness", "Contrast", "Downsample",
         "Sharpen", "Equalize", "ProcessCroppedImage"]

ALL_TYPES = STOCHASTIC + FIXED + ["TemporalShift"]   # 16

GROUP_A = {"normal", "up", "down"}
GROUP_B = {"left", "right", "unknown"}


def pick_doubled(sessions, meta, n_from_A, n_from_B, rng):
    """Choose which sessions get a second augmented set."""
    a = [s for s in sessions if meta[s] in GROUP_A]
    b = [s for s in sessions if meta[s] in GROUP_B]
    return rng.sample(a, n_from_A) + rng.sample(b, n_from_B)


def build_plan(sessions, meta, split, n_needed, rng):
    """Return list of (source_session, aug_type) with no session+type repeat."""
    if split == "train":
        types = [t for t in ALL_TYPES for _ in range(3)]      # 16 x 3 = 48
        doubled = pick_doubled(sessions, meta, 3, 3, rng)
    else:
        picks = rng.sample(FIXED, 3)
        types = STOCHASTIC + picks                            # 8 + 3 = 11
        doubled = pick_doubled(sessions, meta, 1, 1, rng)

    assert len(types) == n_needed, (len(types), n_needed)

    slots = list(sessions) + doubled                          # each session once, doubled twice
    assert len(slots) == n_needed, (len(slots), n_needed)

    # assign types to slots, avoiding same session getting same type twice
    for attempt in range(200):
        t = types[:]
        rng.shuffle(t)
        used = defaultdict(set)
        ok = True
        pairs = []
        for sess, typ in zip(slots, t):
            if typ in used[sess]:
                ok = False
                break
            used[sess].add(typ)
            pairs.append((sess, typ))
        if ok:
            return pairs
    raise RuntimeError(f"could not assign for {split}")


rng = random.Random(AUG_SEED)
angle_of = dict(zip(df.set_num, df.angle))

plan_rows = []
for split, n_needed in [("train", 48), ("validation", 11), ("test", 11)]:
    sessions = sorted(df.loc[df.split == split, "set_num"].tolist())
    pairs = build_plan(sessions, angle_of, split, n_needed, rng)
    for i, (sess, typ) in enumerate(pairs, start=1):
        plan_rows.append(dict(split=split, aug_index=i,
                              aug_set_name=f"aug_{split}_{i:02d}",
                              source_set=sess,
                              source_name=f"set_{sess:02d}",
                              source_angle=angle_of[sess],
                              aug_type=typ))

plan = pd.DataFrame(plan_rows)

print("counts per split:")
print(plan.split.value_counts(), "\n")

print("TRAIN - augmentation type usage:")
print(plan[plan.split=="train"].aug_type.value_counts().to_dict(), "\n")

for sp in ["validation", "test"]:
    sub = plan[plan.split==sp]
    print(f"{sp.upper()} - types used:", sorted(sub.aug_type.unique()))
    print(f"{sp.upper()} - fixed picks:", sorted(set(sub.aug_type) & set(FIXED)))
    print(f"{sp.upper()} - sessions used twice:",
          sorted(sub.source_set.value_counts()[lambda x: x>1].index.tolist()), "\n")

print("TRAIN - sessions used twice:",
      sorted(plan[plan.split=="train"].source_set.value_counts()[lambda x: x>1].index.tolist()))

dupes = plan.groupby(['source_set','aug_type']).size()
print("\nany session+type repeated:", (dupes > 1).any())

counts per split:
split
train         48
validation    11
test          11
Name: count, dtype: int64 

TRAIN - augmentation type usage:
{'Equalize': 3, 'HorizontalFlip': 3, 'Rotate': 3, 'Scale': 3, 'Downsample': 3, 'TemporalShift': 3, 'OpticalDistortion': 3, 'Sharpen': 3, 'GaussianBlur': 3, 'RGBShift': 3, 'Brightness': 3, 'Translate': 3, 'Perspective': 3, 'ProcessCroppedImage': 3, 'Contrast': 3, 'GaussNoise': 3} 

VALIDATION - types used: ['Contrast', 'GaussNoise', 'GaussianBlur', 'HorizontalFlip', 'OpticalDistortion', 'Perspective', 'RGBShift', 'Rotate', 'Scale', 'Sharpen', 'Translate']
VALIDATION - fixed picks: ['Contrast', 'HorizontalFlip', 'Sharpen']
VALIDATION - sessions used twice: [15, 27] 

TEST - types used: ['Contrast', 'Downsample', 'Equalize', 'GaussNoise', 'GaussianBlur', 'OpticalDistortion', 'Perspective', 'RGBShift', 'Rotate', 'Scale', 'Translate']
TEST - fixed picks: ['Contrast', 'Downsample', 'Equalize']
TEST - sessions used twice: [34, 60] 

TRAIN - sessions used twic

In [10]:
plan.to_csv("/home/admins/rebuild_workspace/logs/augmentation_plan_log.csv", index=False)
print("saved")

saved


In [11]:
def get_augmentations(t):
    a = {
        "HorizontalFlip": Compose([HorizontalFlip(p=1.0)]),
        "Brightness": Compose([RandomBrightnessContrast(brightness_limit=(0.3,0.3), contrast_limit=0.0, p=1.0)]),
        "Contrast": Compose([RandomBrightnessContrast(brightness_limit=0.0, contrast_limit=(0.3,0.3), p=1.0)]),
        "Rotate": Compose([Rotate(limit=5, p=1.0)]),
        "Translate": Compose([Affine(translate_px={"x":(-10,10),"y":(-10,10)}, p=1.0)]),
        "Scale": Compose([Affine(scale=(0.9,1.1), p=1.0)]),
        "GaussianBlur": Compose([GaussianBlur(blur_limit=(7,15), p=1.0)]),
        "GaussNoise": Compose([GaussNoise(var_limit=(100.0,250.0), p=1.0)]),
        "RGBShift": Compose([RGBShift(r_shift_limit=50, g_shift_limit=50, b_shift_limit=50, p=1.0)]),
        "OpticalDistortion": Compose([OpticalDistortion(distort_limit=0.2, shift_limit=0.2, p=1.0)]),
        "Perspective": Compose([Perspective(scale=(0.05,0.1), p=1.0)]),
        "Equalize": Compose([Equalize(p=1.0)]),
    }
    return a.get(t)

def downsample_image(img, scale=0.5):
    h, w = img.shape[:2]
    return cv2.resize(cv2.resize(img, (int(w*scale), int(h*scale))), (w, h))

def sharpen_image(img):
    return cv2.filter2D(img, -1, np.array([[0,-1,0],[-1,5,-1],[0,-1,0]]))

def process_cropped_image(img):
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(3,3)).apply(l)
    out = cv2.cvtColor(cv2.merge((l,a,b)), cv2.COLOR_LAB2BGR)
    out = cv2.GaussianBlur(out, (7,7), 0)
    out = cv2.bilateralFilter(out, d=5, sigmaColor=75, sigmaSpace=75)
    out = cv2.filter2D(out, -1, np.array([[-1,-1,-1],[-1,9,-1],[-1,-1,-1]]))
    return cv2.GaussianBlur(out, (5,5), 0)

def augment_image(img, t):
    if t == "Downsample":          return downsample_image(img)
    if t == "Sharpen":             return sharpen_image(img)
    if t == "TemporalShift":       return img
    if t == "ProcessCroppedImage": return process_cropped_image(img)
    return get_augmentations(t)(image=img)['image']


ALL = ["Rotate","Translate","Scale","GaussianBlur","GaussNoise","RGBShift",
       "OpticalDistortion","Perspective","HorizontalFlip","Brightness","Contrast",
       "Downsample","Sharpen","Equalize","ProcessCroppedImage","TemporalShift"]

sample = cv2.imread("/home/admins/rebuild_workspace/02_frames_cropped/set_01/bat/01.png")
print("input:", sample.shape)

for t in ALL:
    try:
        a = augment_image(sample.copy(), t)
        b = augment_image(sample.copy(), t)
        same_shape = a.shape == sample.shape
        identical  = np.array_equal(a, b)
        diff = np.abs(a.astype(int) - sample.astype(int)).mean()
        print(f"{t:22s} shape_ok={same_shape}  repeat_identical={identical}  mean_change={diff:6.2f}")
    except Exception as e:
        print(f"{t:22s} ERROR: {e}")

input: (80, 112, 3)
Rotate                 shape_ok=True  repeat_identical=False  mean_change=  5.94
Translate              shape_ok=True  repeat_identical=False  mean_change= 31.83
Scale                  shape_ok=True  repeat_identical=False  mean_change=  2.09
GaussianBlur           shape_ok=True  repeat_identical=False  mean_change=  2.83
GaussNoise             shape_ok=True  repeat_identical=False  mean_change= 11.29
RGBShift               shape_ok=True  repeat_identical=False  mean_change= 19.67
OpticalDistortion      shape_ok=True  repeat_identical=False  mean_change=  3.45
Perspective            shape_ok=True  repeat_identical=False  mean_change= 15.67
HorizontalFlip         shape_ok=True  repeat_identical=True  mean_change= 21.12
Brightness             shape_ok=True  repeat_identical=True  mean_change= 75.50
Contrast               shape_ok=True  repeat_identical=True  mean_change= 38.93
Downsample             shape_ok=True  repeat_identical=True  mean_change=  2.08
Sharpen     

In [12]:
WORKSPACE   = "/home/admins/rebuild_workspace"
CROPPED_DIR = os.path.join(WORKSPACE, "02_frames_cropped")
AUG_DIR     = os.path.join(WORKSPACE, "03_frames_augmented")
LOGS_DIR    = os.path.join(WORKSPACE, "logs")

TARGET_SIZE = (112, 80)     # (width, height)
WORDS = ['bat','cup','drop','eat','fish','hot','jump','milk','pen','red']

plan = pd.read_csv(os.path.join(LOGS_DIR, "augmentation_plan_log.csv"))


def resize_if_needed(img):
    h, w = img.shape[:2]
    tw, th = TARGET_SIZE
    if (h, w) == (th, tw):
        return img, False
    interp = cv2.INTER_AREA if (h > th or w > tw) else cv2.INTER_LINEAR
    return cv2.resize(img, (tw, th), interpolation=interp), True


def make_one_set(row):
    """Generate all 10 word folders for one augmented set."""
    src_set  = row['source_name']
    aug_name = row['aug_set_name']
    aug_type = row['aug_type']
    resized_count = 0

    for word in WORDS:
        src = os.path.join(CROPPED_DIR, src_set, word)
        dst = os.path.join(AUG_DIR, aug_name, word)
        os.makedirs(dst, exist_ok=True)

        files = sorted(f for f in os.listdir(src) if f.endswith('.png'))
        for idx, f in enumerate(files, start=1):
            img = cv2.imread(os.path.join(src, f))
            out = augment_image(img, aug_type)
            out, was_resized = resize_if_needed(out)
            if was_resized:
                resized_count += 1
            cv2.imwrite(os.path.join(dst, f"{idx:02d}.png"), out)

    return dict(aug_set_name=aug_name, source_set=src_set,
                aug_type=aug_type, split=row['split'],
                frames_written=len(WORDS) * 60,
                frames_resized=resized_count)


if True:
    rows = plan.to_dict('records')
    results, start, done = [], datetime.now(), 0

    with ProcessPoolExecutor(max_workers=10) as ex:
        futures = {ex.submit(make_one_set, r): r for r in rows}
        for fut in as_completed(futures):
            results.append(fut.result())
            done += 1
            if done % 10 == 0:
                print(f"{done}/{len(rows)}  ({(datetime.now()-start).seconds}s)")

    gen = pd.DataFrame(results).sort_values(['split','aug_set_name']).reset_index(drop=True)
    gen.to_csv(os.path.join(LOGS_DIR, "augmentation_generated_log.csv"), index=False)

    print("\n--- summary ---")
    print("augmented sets:", len(gen))
    print("per split:", gen.split.value_counts().to_dict())
    print("total frames:", gen.frames_written.sum())
    print("frames needing resize:", gen.frames_resized.sum())
    print("\ntype usage per split:")
    print(pd.crosstab(gen.aug_type, gen.split))
    print("\nelapsed:", (datetime.now()-start).seconds, "s")

10/70  (3s)
20/70  (7s)
30/70  (10s)
40/70  (12s)
50/70  (16s)
60/70  (18s)
70/70  (21s)

--- summary ---
augmented sets: 70
per split: {'train': 48, 'test': 11, 'validation': 11}
total frames: 42000
frames needing resize: 0

type usage per split:
split                test  train  validation
aug_type                                    
Brightness              0      3           0
Contrast                1      3           1
Downsample              1      3           0
Equalize                1      3           0
GaussNoise              1      3           1
GaussianBlur            1      3           1
HorizontalFlip          0      3           1
OpticalDistortion       1      3           1
Perspective             1      3           1
ProcessCroppedImage     0      3           0
RGBShift                1      3           1
Rotate                  1      3           1
Scale                   1      3           1
Sharpen                 0      3           1
TemporalShift           0      3

In [13]:
WORKSPACE   = "/home/admins/rebuild_workspace"
CROPPED_DIR = os.path.join(WORKSPACE, "02_frames_cropped")
AUG_DIR     = os.path.join(WORKSPACE, "03_frames_augmented")
GRID_DIR    = os.path.join(WORKSPACE, "04_grids_combined")
LOGS_DIR    = os.path.join(WORKSPACE, "logs")

ROWS, COLS = 10, 6
WORDS = ['bat','cup','drop','eat','fish','hot','jump','milk','pen','red']


def build_grid(args):
    """Assemble one 10x6 grid for a single set+word."""
    src_dir, set_label, word = args
    word_path = os.path.join(src_dir, set_label, word)

    frames = sorted(f for f in os.listdir(word_path) if f.endswith('.png'))
    if len(frames) != ROWS * COLS:
        return dict(set_label=set_label, word=word, status=f"WRONG_COUNT_{len(frames)}",
                    height=0, width=0)

    first = cv2.imread(os.path.join(word_path, frames[0]))
    fh, fw, ch = first.shape
    grid = np.zeros((fh * ROWS, fw * COLS, ch), dtype=np.uint8)

    for idx, f in enumerate(frames):
        img = cv2.imread(os.path.join(word_path, f))
        if img is None:
            return dict(set_label=set_label, word=word, status=f"READ_FAIL_{f}",
                        height=0, width=0)
        r, c = idx // COLS, idx % COLS
        grid[r*fh:(r+1)*fh, c*fw:(c+1)*fw] = img

    os.makedirs(GRID_DIR, exist_ok=True)
    out_name = f"{set_label}_{word}.png"
    cv2.imwrite(os.path.join(GRID_DIR, out_name), grid)

    return dict(set_label=set_label, word=word, status="OK",
                height=grid.shape[0], width=grid.shape[1])


if True:
    tasks = []
    for s in sorted(os.listdir(CROPPED_DIR)):
        for w in WORDS:
            tasks.append((CROPPED_DIR, s, w))
    for s in sorted(os.listdir(AUG_DIR)):
        for w in WORDS:
            tasks.append((AUG_DIR, s, w))

    print("grids to build:", len(tasks))

    results, start, done = [], datetime.now(), 0
    with ProcessPoolExecutor(max_workers=10) as ex:
        futures = [ex.submit(build_grid, t) for t in tasks]
        for fut in as_completed(futures):
            r = fut.result()
            results.append(r)
            done += 1
            if r['status'] != "OK":
                print(f"  PROBLEM {r['set_label']}/{r['word']}: {r['status']}")
            if done % 200 == 0:
                print(f"{done}/{len(tasks)}  ({(datetime.now()-start).seconds}s)")

    log = pd.DataFrame(results).sort_values(['set_label','word']).reset_index(drop=True)
    log.to_csv(os.path.join(LOGS_DIR, "grid_assembly_log.csv"), index=False)

    print("\n--- summary ---")
    print("grids built:", (log.status == "OK").sum(), "of", len(log))
    print("problems:", (log.status != "OK").sum())
    print("dimensions:", log[log.status=="OK"].groupby(['height','width']).size().to_dict())
    print("files on disk:", len(os.listdir(GRID_DIR)))
    print("elapsed:", (datetime.now()-start).seconds, "s")

grids to build: 1300
200/1300  (1s)
400/1300  (2s)
600/1300  (3s)
800/1300  (4s)
1000/1300  (5s)
1200/1300  (5s)

--- summary ---
grids built: 1300 of 1300
problems: 0
dimensions: {(800, 672): 1300}
files on disk: 1300
elapsed: 6 s


In [14]:
WORKSPACE    = "/home/admins/rebuild_workspace"
GRID_DIR     = os.path.join(WORKSPACE, "04_grids_combined")
RESIZED_DIR  = os.path.join(WORKSPACE, "05_grids_resized")
LOGS_DIR     = os.path.join(WORKSPACE, "logs")

NEW_SIZE = (224, 224)

os.makedirs(RESIZED_DIR, exist_ok=True)


def resize_one(fname):
    src = os.path.join(GRID_DIR, fname)
    dst = os.path.join(RESIZED_DIR, fname)
    try:
        with Image.open(src) as img:
            orig = img.size
            out = img.resize(NEW_SIZE, Image.LANCZOS)
            out.save(dst)
        return dict(filename=fname, orig_w=orig[0], orig_h=orig[1], status="OK")
    except Exception as e:
        return dict(filename=fname, orig_w=0, orig_h=0, status=f"ERROR: {e}")


if True:
    files = sorted(f for f in os.listdir(GRID_DIR) if f.endswith('.png'))
    print("images to resize:", len(files))

    results, start, done = [], datetime.now(), 0
    with ProcessPoolExecutor(max_workers=10) as ex:
        futures = [ex.submit(resize_one, f) for f in files]
        for fut in as_completed(futures):
            r = fut.result()
            results.append(r)
            done += 1
            if r['status'] != "OK":
                print(" ", r['filename'], r['status'])
            if done % 300 == 0:
                print(f"{done}/{len(files)}  ({(datetime.now()-start).seconds}s)")

    log = pd.DataFrame(results).sort_values('filename').reset_index(drop=True)
    log.to_csv(os.path.join(LOGS_DIR, "resize_log.csv"), index=False)

    print("\n--- summary ---")
    print("resized:", (log.status == "OK").sum(), "of", len(log))
    print("source dimensions:", log[log.status=="OK"].groupby(['orig_w','orig_h']).size().to_dict())
    print("files on disk:", len(os.listdir(RESIZED_DIR)))
    print("elapsed:", (datetime.now()-start).seconds, "s")

images to resize: 1300
300/1300  (1s)
600/1300  (2s)
900/1300  (3s)
1200/1300  (4s)

--- summary ---
resized: 1300 of 1300
source dimensions: {(672, 800): 1300}
files on disk: 1300
elapsed: 5 s


In [15]:
WORKSPACE   = "/home/admins/rebuild_workspace"
RESIZED_DIR = os.path.join(WORKSPACE, "05_grids_resized")
FINAL_DIR   = os.path.join(WORKSPACE, "06_dataset_final")
LOGS_DIR    = os.path.join(WORKSPACE, "logs")
SPLIT_CSV   = "/home/admins/lip_codebase_clean/docs/results_rebuilt_data/session_split_assignment.csv"

WORDS = ['bat','cup','drop','eat','fish','hot','jump','milk','pen','red']
SPLIT_FOLDER = {"train": "Train", "validation": "Validation", "test": "Test"}

split_df = pd.read_csv(SPLIT_CSV)
real_split = dict(zip(split_df.folder, split_df.split))   # 'set_01' -> 'train'


def classify(fname):
    """Return (split, source_type, set_label, word) for a grid filename."""
    stem = fname[:-4]
    for w in WORDS:
        if stem.endswith("_" + w):
            set_label = stem[:-(len(w) + 1)]
            word = w
            break
    else:
        return None

    if set_label.startswith("aug_"):
        m = re.match(r"aug_(train|validation|test)_\d+$", set_label)
        if not m:
            return None
        return m.group(1), "augmented", set_label, word

    if set_label in real_split:
        return real_split[set_label], "real", set_label, word

    return None


if True:
    files = sorted(f for f in os.listdir(RESIZED_DIR) if f.endswith('.png'))

    # group by (split, word), real first then augmented, so numbering is stable
    buckets = {}
    unclassified = []
    for f in files:
        info = classify(f)
        if info is None:
            unclassified.append(f)
            continue
        split, src_type, set_label, word = info
        buckets.setdefault((split, word), []).append((src_type, set_label, f))

    if unclassified:
        print("UNCLASSIFIED:", unclassified)

    for k in buckets:
        buckets[k].sort(key=lambda t: (t[0] != "real", t[1]))   # real first, then by label

    records = []
    for (split, word), items in sorted(buckets.items()):
        dest = os.path.join(FINAL_DIR, SPLIT_FOLDER[split], word)
        os.makedirs(dest, exist_ok=True)
        for i, (src_type, set_label, fname) in enumerate(items, start=1):
            new_name = f"{i:03d}.png"
            shutil.copy2(os.path.join(RESIZED_DIR, fname), os.path.join(dest, new_name))
            records.append(dict(split=split, word=word, final_number=i,
                                final_name=new_name, source_type=src_type,
                                source_set=set_label, source_file=fname))

    mapping = pd.DataFrame(records)
    mapping.to_csv(os.path.join(LOGS_DIR, "final_image_mapping_log.csv"), index=False)

    print("\n--- summary ---")
    print("images placed:", len(mapping))
    print()
    print(pd.crosstab(mapping.split, mapping.source_type))
    print()
    counts = mapping.groupby(['split','word']).size().unstack()
    print("per word per split:")
    print(counts)
    print()
    for sp, folder in SPLIT_FOLDER.items():
        p = os.path.join(FINAL_DIR, folder)
        n = {w: len(os.listdir(os.path.join(p, w))) for w in WORDS}
        print(f"{folder}: {n}")


--- summary ---
images placed: 1300

source_type  augmented  real
split                       
test               110    90
train              480   420
validation         110    90

per word per split:
word        bat  cup  drop  eat  fish  hot  jump  milk  pen  red
split                                                           
test         20   20    20   20    20   20    20    20   20   20
train        90   90    90   90    90   90    90    90   90   90
validation   20   20    20   20    20   20    20    20   20   20

Train: {'bat': 90, 'cup': 90, 'drop': 90, 'eat': 90, 'fish': 90, 'hot': 90, 'jump': 90, 'milk': 90, 'pen': 90, 'red': 90}
Validation: {'bat': 20, 'cup': 20, 'drop': 20, 'eat': 20, 'fish': 20, 'hot': 20, 'jump': 20, 'milk': 20, 'pen': 20, 'red': 20}
Test: {'bat': 20, 'cup': 20, 'drop': 20, 'eat': 20, 'fish': 20, 'hot': 20, 'jump': 20, 'milk': 20, 'pen': 20, 'red': 20}


In [16]:
map = pd.read_csv("/home/admins/rebuild_workspace/logs/final_image_mapping_log.csv")
plan = pd.read_csv("/home/admins/rebuild_workspace/logs/augmentation_plan_log.csv")
split_df = pd.read_csv("/home/admins/lip_codebase_clean/docs/results_rebuilt_data/session_split_assignment.csv")

# map every final image back to the ORIGINAL recording session
aug_source = dict(zip(plan.aug_set_name, plan.source_name))
map['origin_session'] = map.apply(
    lambda r: r.source_set if r.source_type == 'real' else aug_source[r.source_set], axis=1)

# which splits does each origin session appear in?
leak = map.groupby('origin_session')['split'].nunique()
print("sessions appearing in more than one split:", (leak > 1).sum())
print()
print("sessions per split:")
print(map.groupby('split')['origin_session'].nunique())

sessions appearing in more than one split: 0

sessions per split:
split
test           9
train         42
validation     9
Name: origin_session, dtype: int64
